# Analisis Exploratorio de Datos

Finalmente con la herramienta de descarga podemos entrar al siguiente apartado. El analisis exploratorio de datos.

## Agro en acción

Denominando Agro a nuestra herramienta para la descarga de archivos del DGSIAP. Haremos una prueba.

Supongamos que queremos la información de la producción agricola municipal de todos los años disponibles

In [1]:
# Importamos el diccionario maestro. (No olvide elegir su categoria)
from scripts import dgsiap

# Importamos las herramientas para descarga
from scripts import download as d 

# Informacion municipal historica
request = {
    "category":"Municipal",
    "period": "*"}

client = dgsiap.dict_master()

# Generamos nuestra solicitud
dic_link = d.downassist(request=request, ref=client)

# Descargamos los archivos requeridos
d.downbot(dic_link)

Bienvenido al script de descarga de datos abiertos del DGSIAP.
Conexion Establecida
Usted escogio Agricola
Los datos historicos se presentan en el espacio denomiado: Información por nivel
Los datos se encuentran divididos en 3 bloques:

1. Municipal
2. Nacional
3. No seguimiento
El archivo ya fue descargado. Se encuentra en /data/external/Diccionario_agricola_2003_a_2025.xlsx
El archivo ya fue descargado. Se encuentra en /data/external/Cierre_agricola_mun_2025.csv
El archivo ya fue descargado. Se encuentra en /data/external/Cierre_agricola_mun_2024.csv
El archivo ya fue descargado. Se encuentra en /data/external/Cierre_agricola_mun_2023.csv
El archivo ya fue descargado. Se encuentra en /data/external/Cierre_agricola_mun_2022.csv
El archivo ya fue descargado. Se encuentra en /data/external/Cierre_agricola_mun_2021.csv
El archivo ya fue descargado. Se encuentra en /data/external/Cierre_agricola_mun_2020.csv
El archivo ya fue descargado. Se encuentra en /data/external/Cierre_agricola_mun_

# Formato

Con agro podemos descargar los archivos que necesitemos para el analisis. Sin emabargo, debemos asegurarnos de no perder información al momento de descargar la información. No confundir con los datos faltantes. Estos se originaron por la naturaleza de la fuente. Nuestro problema es con respecto a información que si estaba pero perdimos al momento de cargar la información.

## Encoding.

Uno de los problemas más conocidos al momento de descargar información es el desconocimiento del enconding usado al generar el conjunto de datos. Por defecto procuraremos usar el formato estándar UTF-8. Sin embargo, es posible encontrar archivos con coficiaciones distintas. Entonces debemos buscar la forma de leerlas con la minima pérdida

In [18]:
import cchardet
from pathlib import Path
import os

# Tomamos algun archivo .csv

csv_set = list(DATA_PATH.glob("*.csv"))
csv = csv_set[0]

try:
    pd.read_csv(csv)
except UnicodeDecodeError as e:
    print(e)

'utf-8' codec can't decode byte 0xf1 in position 350: invalid continuation byte


Como podemos observar, ni siquiera es posible leerlo. Entonces debemos encontrar su codificación para poder leerlo. Para ello usaremos la libreria `cchardet`.

In [19]:
with open(csv, 'rb') as f: # Leemos su archivo binario asociado
    data = f.read()
    encode = cchardet.detect(data)

print(encode)

{'encoding': 'ISO-8859-1', 'confidence': 0.8167673945426941, 'language': 'es'}


Viendo que nos devuelve el encoding que la libreria detecto, además de 2 valores más

* confidence: Un valor entre 0 y 1 que define la seguridad de la respuesta.
* language: el idioma que detecto. (Este puede ser importante si nos detecta un idioma no relacionado con el documento)

Entonces aplicamos lo anterior al resto de documentos.

In [30]:
dgsiap_encode = {}

def get_encoding(filepath):
    with open(filepath, 'rb') as file:
        data=file.read()
        res = cchardet.detect(data)
    return res

for c in csv_set:
    filename = str(c).split("/")[-1]
    dgsiap_encode[filename] = get_encoding(c)

Algo a comentar de la libreria es que es algo lenta para encontrarlos. Sin embargo, lo esperado es hacer esto durante la descarga de los archivos. Asi cuando las descargas terminen tener todo listo para el analisis.

In [31]:
dgsiap_encode

{'Cierre_agricola_mun_2025.csv': {'encoding': 'ISO-8859-1',
  'confidence': 0.8167673945426941,
  'language': 'es'},
 'Cierre_agricola_mun_2024.csv': {'encoding': 'ISO-8859-1',
  'confidence': 0.8172728419303894,
  'language': 'es'},
 'Cierre_agricola_mun_2023.csv': {'encoding': 'ISO-8859-1',
  'confidence': 0.8171019554138184,
  'language': 'es'},
 'Cierre_agricola_mun_2022.csv': {'encoding': 'ISO-8859-1',
  'confidence': 0.8172381520271301,
  'language': 'es'},
 'Cierre_agricola_mun_2021.csv': {'encoding': 'ISO-8859-1',
  'confidence': 0.8153364062309265,
  'language': 'es'},
 'Cierre_agricola_mun_2020.csv': {'encoding': 'ISO-8859-1',
  'confidence': 0.8148902058601379,
  'language': 'es'},
 'Cierre_agricola_mun_2019.csv': {'encoding': 'ISO-8859-1',
  'confidence': 0.8173798322677612,
  'language': 'es'},
 'Cierre_agricola_mun_2018.csv': {'encoding': 'ISO-8859-1',
  'confidence': 0.8176097869873047,
  'language': 'es'},
 'Cierre_agricola_mun_2017.csv': {'encoding': 'ISO-8859-1',
  'c

Hacemos la prueba final. Intentamos leer todos los csv.

In [35]:
for c in csv_set:
    filename = str(c).split("/")[-1]
    code = dgsiap_encode[filename]
    try:
        pd.read_csv(c, encoding= code['encoding'], low_memory=False)
    except UnicodeDecodeError as e:
        print(filename, e)


Entonces ya es posible leer cualquier archivo .csv mediante pandas. Sin embargo, esto no garantiza que no existan errores a futuro que debamos corregir durante el analisis. La prioridad de esta sección es que cualquier archivo pueda ser consultado para el analisis.

In [38]:
csv_sample = csv_set[0]

test = pd.read_csv(csv_sample, encoding= code['encoding'], low_memory=False)
print(test)


       Anio  Idestado       Nomestado  Idddr          Nomddr  Idcader  \
0      2025         1  Aguascalientes      1  Aguascalientes        1   
1      2025         1  Aguascalientes      1  Aguascalientes        1   
2      2025         1  Aguascalientes      1  Aguascalientes        1   
3      2025         1  Aguascalientes      1  Aguascalientes        1   
4      2025         1  Aguascalientes      1  Aguascalientes        1   
...     ...       ...             ...    ...             ...      ...   
35897  2025        32       Zacatecas    190           Jalpa        4   
35898  2025        32       Zacatecas    190           Jalpa        4   
35899  2025        32       Zacatecas    190           Jalpa        4   
35900  2025        32       Zacatecas    190           Jalpa        4   
35901  2025        32       Zacatecas    190           Jalpa        4   

             Nomcader  Idmunicipio    Nommunicipio  Idciclo  ... Nomunidad  \
0      Aguascalientes            1  Aguascali

En primera instancia se ve bien, pero no debemos bajar la guardia. Sin embargo, los detalles pueden esperar un momento. Ahora tenemos otro problema. Entender que hace cada variable del documento. Aqui es donde entran los diccionarios de datos.

## Diccionario de Datos

In [55]:

print(csv_sample) # El url nos da información del diccionario de datos asociado.
print(dic_link)  # Revisamos la información solicitada

dicts = list(DATA_PATH.glob("*.xlsx"))
print(dicts)

# Esto se puede hacer más limbio con un diccionario de archivos descargados. 

v = dic_link['Municipal']['Codebook']
print("Link", v)  # Obtenemos el url de donde vino.
print(d.get_filename(d.parse_link(v)))
dictname = d.get_filename(d.parse_link(v))
codebook = pd.read_excel(DATA_PATH / dictname)

print(codebook.head(20))

../data/external/Cierre_agricola_mun_2025.csv
{'Municipal': {'Codebook': '/index.php?view=5D24F995-E734D711-1EA6F8D2-9D1387D7', 'Data': {'2025': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2025'), '2024': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2024'), '2023': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2023'), '2022': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2022'), '2021': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2021'), '2020': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2020'), '2019': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2019'), '2018': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2018'), '2017': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2017'), '2016': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2016'), '2015': URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2015'), '2014